In [1]:
# Cell 1: Download trained model files from GitHub
!wget -q https://raw.githubusercontent.com/singhkhushi0115/rice-pest-identification/main/rice_pest_model.pth -O rice_pest_model.pth
!wget -q https://raw.githubusercontent.com/singhkhushi0115/rice-pest-identification/main/class_names.json -O class_names.json

!ls -la rice_pest_model.pth class_names.json

-rw-r--r-- 1 root root      307 Jul 24 17:03 class_names.json
-rw-r--r-- 1 root root 44815051 Jul 24 17:03 rice_pest_model.pth


In [2]:
# Cell 2: Install dependencies and load the trained model
!pip install gradio --quiet

import torch
import torch.nn as nn
from torchvision import models, transforms
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load class names
with open("class_names.json") as f:
    class_names = json.load(f)
print("Classes:", class_names)

# Rebuild the same architecture used during training
model = models.resnet18(weights=None)  # no pretrained weights needed — we're loading our own
model.fc = nn.Linear(model.fc.in_features, len(class_names))

# Load trained weights
model.load_state_dict(torch.load("rice_pest_model.pth", map_location=device))
model = model.to(device)
model.eval()

print("Model loaded successfully.")

Using device: cpu
Classes: ['Rice Stemfly', 'asiatic rice borer', 'brown plant hopper', 'grain spreader thrips', 'paddy stem maggot', 'rice gall midge', 'rice leaf caterpillar', 'rice leaf roller', 'rice leafhopper', 'rice shell pest', 'rice water weevil', 'small brown plant hopper', 'white backed plant hopper', 'yellow rice borer']
Model loaded successfully.


In [3]:
# Cell 3: Image preprocessing + prediction function
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

recommendations = {
    "rice leaf roller": "Remove and destroy affected leaves; apply neem-based biopesticide early.",
    "rice leaf caterpillar": "Handpick larvae if infestation is small; use recommended insecticide for larger fields.",
    "paddy stem maggot": "Drain field temporarily to reduce larvae survival; consult local agri office for chemical control.",
    "asiatic rice borer": "Use pheromone traps; apply carbofuran granules if infestation crosses threshold.",
    "yellow rice borer": "Encourage natural predators (spiders, wasps); apply targeted insecticide during egg-hatch period.",
    "rice gall midge": "Use resistant rice varieties next season; remove and destroy galled tillers.",
    "Rice Stemfly": "Maintain field hygiene; apply systemic insecticide if damage exceeds 10% of tillers.",
    "brown plant hopper": "Avoid excess nitrogen fertilizer; use resistant varieties and monitor with light traps.",
    "white backed plant hopper": "Drain field periodically; apply recommended insecticide at nymph stage.",
    "small brown plant hopper": "Monitor with sticky traps; avoid continuous flooding of the field.",
    "rice water weevil": "Drain field to expose larvae; apply seed treatment before next planting.",
    "rice leafhopper": "Use yellow sticky traps; apply insecticide only if hopper density is high.",
    "grain spreader thrips": "Maintain field moisture; apply recommended miticide/insecticide combination.",
    "rice shell pest": "Remove infested grains promptly; consult agricultural extension for targeted treatment."
}

def predict_pest(image):
    img = image.convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)
        conf, pred_idx = torch.max(probs, 1)

    pest_name = class_names[pred_idx.item()]
    confidence = conf.item() * 100
    recommendation = recommendations.get(pest_name, "Consult local agricultural expert for specific treatment.")

    result = f"**Predicted Pest:** {pest_name}\n\n**Confidence:** {confidence:.1f}%\n\n**Recommended Action:** {recommendation}"
    return result

print("Prediction function ready.")

Prediction function ready.


In [4]:
# Cell 4: Launch standalone Gradio app
import gradio as gr

demo = gr.Interface(
    fn=predict_pest,
    inputs=gr.Image(type="pil", label="Upload a Rice Pest Image"),
    outputs=gr.Markdown(label="Prediction & Recommendation"),
    title="🌾 Rice Pest Identification System",
    description="Upload an image of a pest found on rice crops to identify the species and get a recommended action. (This app runs independently — it only loads a pre-trained model, no training happens here.)"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8c21d1fb52142726a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# Cell 5 (enhanced final version): Custom-styled Gradio interface with rich result card
import gradio as gr

custom_css = """
.gradio-container {
    background: linear-gradient(180deg, #e8f5e9 0%, #f4f9f4 100%);
    font-family: 'Poppins', 'Segoe UI', Arial, sans-serif;
}

#header-box {
    background: linear-gradient(120deg, #1b5e20, #43a047, #7cb342);
    padding: 32px 20px;
    border-radius: 20px;
    color: white;
    text-align: center;
    margin-bottom: 20px;
    box-shadow: 0 8px 24px rgba(27, 94, 32, 0.35);
}
#header-box h1 {
    margin: 0;
    font-size: clamp(22px, 4vw, 34px);
    font-weight: 800;
}
#header-box p {
    margin-top: 8px;
    font-size: clamp(13px, 2vw, 16px);
    opacity: 0.95;
}

#instructions {
    background: #fff3cd !important;
    border-left: 6px solid #f9a825;
    padding: 14px 18px;
    border-radius: 10px;
    font-size: 14px;
    margin-bottom: 20px;
}
#instructions p { color: #4e342e !important; margin: 0; font-weight: 500; }

.gr-button-primary {
    background: linear-gradient(120deg, #2e7d32, #66bb6a) !important;
    border: none !important;
    font-size: 17px !important;
    font-weight: 700 !important;
    padding: 14px !important;
    border-radius: 12px !important;
    box-shadow: 0 4px 14px rgba(46, 125, 50, 0.4) !important;
}

footer {visibility: hidden}

@media (max-width: 768px) {
    #header-box { padding: 20px 14px; }
}
"""

header_html = """
<div id="header-box">
    <h1>🌾 Rice Pest Identification System</h1>
    <p>AI-powered pest detection for rice crops — upload a photo, get an instant identification and action plan.</p>
</div>
"""

instructions_html = """
<div id="instructions">
    <p>📷 <b>Tip:</b> Use a clear, close-up photo of a single pest with good lighting for best results.</p>
</div>
"""

PLACEHOLDER_HTML = "<div style='padding:20px; color:#333; font-size:15px; background:white; border-radius:12px; box-shadow:0 6px 20px rgba(0,0,0,0.08);'>Upload an image and click <b style='color:#1b5e20;'>Identify Pest</b> to see results here.</div>"

def predict_pest_html(image):
    if image is None:
        return PLACEHOLDER_HTML

    img = image.convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)
        conf, pred_idx = torch.max(probs, 1)

    pest_name = class_names[pred_idx.item()]
    confidence = conf.item() * 100
    recommendation = recommendations.get(pest_name, "Consult local agricultural expert for specific treatment.")

    # Color-code based on confidence level
    if confidence >= 70:
        badge_color, bar_color, level = "#2e7d32", "#43a047", "High Confidence"
    elif confidence >= 45:
        badge_color, bar_color, level = "#f9a825", "#fbc02d", "Moderate Confidence"
    else:
        badge_color, bar_color, level = "#c62828", "#e53935", "Low Confidence"

    result_html = f"""
    <div style='background:white; border-radius:14px; padding:22px; box-shadow:0 6px 20px rgba(0,0,0,0.08);'>
        <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:14px;'>
            <span style='font-size:20px; font-weight:700; color:#1b1b1b;'>🐛 {pest_name.title()}</span>
            <span style='background:{badge_color}; color:white; padding:5px 12px; border-radius:20px; font-size:12px; font-weight:600;'>{level}</span>
        </div>

        <div style='margin-bottom:6px; font-size:13px; color:#555;'>Confidence: {confidence:.1f}%</div>
        <div style='background:#eeeeee; border-radius:8px; height:14px; width:100%; overflow:hidden; margin-bottom:18px;'>
            <div style='background:{bar_color}; width:{confidence}%; height:100%; border-radius:8px;'></div>
        </div>

        <div style='background:#f4f9f4; border-left:4px solid #2e7d32; padding:12px 16px; border-radius:8px;'>
            <span style='font-weight:700; color:#1b5e20;'>✅ Recommended Action:</span>
            <p style='margin:6px 0 0 0; color:#333; font-size:14px; line-height:1.5;'>{recommendation}</p>
        </div>
    </div>
    """
    return result_html

with gr.Blocks(css=custom_css, title="Rice Pest Identification System", theme=gr.themes.Soft(primary_hue="green")) as demo:
    gr.HTML(header_html)
    gr.HTML(instructions_html)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="📤 Upload Pest Image", height=320)
            submit_btn = gr.Button("🔍 IDENTIFY PEST", variant="primary", size="lg")
        with gr.Column(scale=1):
            output_html = gr.HTML(value=PLACEHOLDER_HTML)

    submit_btn.click(fn=predict_pest_html, inputs=image_input, outputs=output_html)

    gr.HTML("<p style='text-align:center; color:#7a7a7a; font-size:13px; margin-top:20px;'>🎓 Built as part of the AI GPU Summer Internship — Presidency University × NVIDIA AI Centre of Excellence</p>")

demo.launch(share=True)

/tmp/ipykernel_1193/3873209776.py:116: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="Rice Pest Identification System", theme=gr.themes.Soft(primary_hue="green")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3da99ea38d562898ae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
